In [ ]:
import pandas as pd
p = "../data/cgmacros/CGMacros/CGMacros-032/CGMacros-032.csv"
df = pd.read_csv(p, parse_dates=["Timestamp"])
df.columns = df.columns.str.strip()
print(df.shape)

In [ ]:
print(df["Timestamp"].diff().value_counts().head())
print(df[["Libre GL", "Dexcom GL"]].describe())
print(df["Intensity"].value_counts().head(10))

In [ ]:
cols = ["Timestamp", "Meal Type", "Calories", "Carbs", "Protein", "Fat", "Fiber", "Amount Consumed", "Image path"]
print(df[df["Image path"].notna()][cols].to_string())

In [ ]:
bio = pd.read_csv("../data/cgmacros/CGMacros/bio.csv")
bio.columns = bio.columns.str.strip()
print(sorted(bio["subject"].tolist()))
print(bio["A1c PDL (Lab)"].describe())

In [ ]:
from pathlib import Path

root = Path("../data/cgmacros/CGMacros")
frames = []
for folder in sorted(root.glob("CGMacros-*")):
    csv_path = folder / f"{folder.name}.csv"
    if not csv_path.exists():
        print("CSV nahi mili:", folder.name)
        continue
    d = pd.read_csv(csv_path, parse_dates=["Timestamp"])
    d.columns = d.columns.str.strip()
    d["subject"] = int(folder.name.split("-")[1])
    frames.append(d)

all_df = pd.concat(frames, ignore_index=True)
print(all_df.shape)
print(all_df.columns.tolist())
print("participants:", all_df["subject"].nunique())

In [ ]:
print(set(all_df["subject"]) == set(bio["subject"]))
print("bio mein hai, folder nahi:", set(bio["subject"]) - set(all_df["subject"]))
print("folder hai, bio mein nahi:", set(all_df["subject"]) - set(bio["subject"]))

In [ ]:
summary = all_df.groupby("subject").agg(
    rows=("Timestamp", "count"),
    libre_mean=("Libre GL", "mean"),
    dexcom_mean=("Dexcom GL", "mean"),
    dexcom_max=("Dexcom GL", "max"),
    dexcom_missing=("Dexcom GL", lambda s: s.isna().mean()),
    hr_missing=("HR", lambda s: s.isna().mean()),
    pct_above_140=("Dexcom GL", lambda s: (s > 140).mean()),
    pct_above_180=("Dexcom GL", lambda s: (s > 180).mean()),
).round(2)
print(summary.to_string())

In [ ]:
check = ["Dexcom GL", "HR", "METs", "Intensity", "Steps", "Sugar", "Carbs", "Fiber"]
count = all_df.groupby("subject")[check].count()   # har column mein kitni values bhari hain
print("Kitne participants mein is column ki kam se kam ek value hai (45 mein se):")
print((count > 0).sum())
print()
print("Participants jinke paas METs hai:", sorted(count.index[count["METs"] > 0].tolist()))
print("Participants jinke paas Steps hai:", sorted(count.index[count["Steps"] > 0].tolist()))

In [ ]:
out = []
for sid, g in all_df.sort_values(["subject", "Timestamp"]).groupby("subject"):
    s = pd.Series(g["Dexcom GL"].values)
    fmax = s[::-1].rolling(120, min_periods=60).max()[::-1].shift(-1)   # agle 120 minute ka max
    ok = s.notna() & fmax.notna()
    row = {"subject": sid}
    for thr in (140, 160, 180):
        row[f"new_{thr}"] = ((s < thr) & (fmax >= thr))[ok].mean()
    row["rise_40"] = ((fmax - s) >= 40)[ok].mean()
    out.append(row)

prev = pd.DataFrame(out).set_index("subject").round(3)
print(prev.to_string())
print()
print("Kitne participants mein kam se kam 1% positive examples hain:")
print((prev >= 0.01).sum())

In [ ]:
cols = ["Dexcom GL", "HR", "Calories (Activity)", "METs", "Intensity", "Steps", "Sugar"]
cov = all_df.groupby("subject")[cols].apply(lambda g: g.notna().mean()).round(2)

print("Kitne participants mein column kam se kam 50% bhara hua hai (45 mein se):")
print((cov > 0.5).sum())
print()
print("METs aur Intensity ka combination (True = 50% se zyada bhara):")
print(pd.DataFrame({"METs": cov["METs"] > 0.5, "Intensity": cov["Intensity"] > 0.5}).value_counts())
print()
print(all_df["METs"].describe())
print(all_df["Sugar"].describe())

In [ ]:
bio["group"] = pd.cut(bio["A1c PDL (Lab)"], bins=[0, 5.7, 6.5, 20],
                      right=False, labels=["normal", "prediabetes", "diabetes"])
tmp = prev.join(bio.set_index("subject")["group"])

print("Participants per group:")
print(tmp.groupby("group", observed=True).size())
print()
print("Event rate (average fraction) by group:")
print(tmp.groupby("group", observed=True)[["new_140", "new_160", "new_180", "rise_40"]].mean().round(3))
print()
print("Participants with fewer than 1% examples of new_140:", (prev["new_140"] < 0.01).sum())
print(prev["new_140"].describe().round(3))

In [ ]:
import pandas as pd
from pathlib import Path

THRESHOLD = 180                      # target ki value, yahin se badlegi

root = Path("../data/cgmacros/CGMacros")

frames = []
for folder in sorted(root.glob("CGMacros-*")):
    d = pd.read_csv(folder / f"{folder.name}.csv", parse_dates=["Timestamp"])
    d.columns = d.columns.str.strip()
    d["subject"] = int(folder.name.split("-")[1])    # folder ke naam se participant ka number
    frames.append(d)
all_df = pd.concat(frames, ignore_index=True)        # 45 files ko ek table mein joda

bio = pd.read_csv(root / "bio.csv")
bio.columns = bio.columns.str.strip()

print(all_df.shape, all_df["subject"].nunique(), bio.shape)

In [ ]:
import matplotlib.pyplot as plt

sid = 5                                   # participant number
g = all_df[all_df["subject"] == sid].sort_values("Timestamp")
day = g.iloc[1440*3 : 1440*4]             # 4th din (1 din = 1440 minute)

plt.figure(figsize=(12, 4))
plt.plot(day["Timestamp"], day["Dexcom GL"])
plt.axhline(THRESHOLD, color="red", linestyle="--")
plt.title(f"Participant {sid}, ek din ka glucose")
plt.ylabel("mg/dL")
plt.show()

In [ ]:
meals = day[day["Meal Type"].notna() & (day["Calories"] > 0)]

plt.figure(figsize=(12, 4))
plt.plot(day["Timestamp"], day["Dexcom GL"])
plt.axhline(THRESHOLD, color="red", linestyle="--")
for t in meals["Timestamp"]:
    plt.axvline(t, color="green", linestyle=":")
plt.title(f"Participant {sid}: glucose aur meals")
plt.ylabel("mg/dL")
plt.show()

print(meals[["Timestamp", "Meal Type", "Carbs", "Fiber", "Calories"]])

In [ ]:
import numpy as np

s = g.set_index("Timestamp")["Dexcom GL"]                       # glucose, time ke hisaab se
future_max = s[::-1].rolling(120, min_periods=60).max()[::-1].shift(-1)   # agle 120 minute ka sabse bada glucose

label = ((s < THRESHOLD) & (future_max >= THRESHOLD)).astype(float)
label[s.isna() | future_max.isna()] = np.nan                    # jahan pata nahi, wahan khali

print(label.value_counts(dropna=False))
print("positive rate:", round(label.mean(), 3))

sd, ld = s.iloc[1440*3 : 1440*4], label.iloc[1440*3 : 1440*4]
plt.figure(figsize=(12, 4))
plt.plot(sd.index, sd.values)
plt.axhline(THRESHOLD, color="red", linestyle="--")
plt.scatter(sd.index[ld == 1], sd[ld == 1], s=8, color="orange")
plt.title("Orange points = label 1 (agle 2 ghante mein 180 cross hoga)")
plt.show()

In [ ]:
gi = g.set_index("Timestamp")
is_meal = gi["Meal Type"].notna() & (gi["Calories"] > 0)
meal_time = gi.index.to_series().where(is_meal).ffill()          # last meal ka time
mins_since_meal = (gi.index.to_series() - meal_time).dt.total_seconds() / 60
meal_carbs = gi["Carbs"].where(is_meal).ffill()                   # last meal ke carbs

df5 = pd.DataFrame({
    "glu_now": s,
    "glu_change_30": s - s.shift(30),     # pichhle 30 minute mein glucose kitna badla
    "mins_since_meal": mins_since_meal,
    "meal_carbs": meal_carbs,
    "label": label,
}).dropna(subset=["label"])

df5 = df5[df5["glu_now"] < THRESHOLD]     # sawaal tabhi, jab abhi 180 se neeche ho

print(df5.groupby("label").size())
print(df5.groupby("label").mean().round(1))

In [ ]:
import pandas as pd
from pathlib import Path

THRESHOLD = 180  # target: glucose crossing this value within the next 2 hours

root = Path("../data/cgmacros/CGMacros")

# Read every participant's file and stack them into one table
frames = []
for folder in sorted(root.glob("CGMacros-*")):
    d = pd.read_csv(folder / f"{folder.name}.csv", parse_dates=["Timestamp"])
    d.columns = d.columns.str.strip()                # remove stray spaces in column names
    d["subject"] = int(folder.name.split("-")[1])    # participant id comes from the folder name
    frames.append(d)
all_df = pd.concat(frames, ignore_index=True)

# Static data: one row per participant
bio = pd.read_csv(root / "bio.csv")
bio.columns = bio.columns.str.strip()

print(all_df.shape, all_df["subject"].nunique(), bio.shape)

In [ ]:
import numpy as np

def build_rows(g, threshold=THRESHOLD):
    """Build one participant's rows: past-only features plus the label."""
    gi = g.sort_values("Timestamp").set_index("Timestamp")
    s = gi["Dexcom GL"]

    # Label: does glucose reach the threshold within the next 120 minutes?
    future_max = s[::-1].rolling(120, min_periods=60).max()[::-1].shift(-1)
    label = ((s < threshold) & (future_max >= threshold)).astype(float)
    label[s.isna() | future_max.isna()] = np.nan   # unknown label

    # Features use only information from the past
    is_meal = gi["Meal Type"].notna() & (gi["Calories"] > 0)
    ts = gi.index.to_series()
    meal_time = ts.where(is_meal).ffill()           # time of the last meal

    out = pd.DataFrame({
        "subject": gi["subject"],
        "glu_now": s,
        "glu_change_30": s - s.shift(30),           # change over the last 30 minutes
        "mins_since_meal": (ts - meal_time).dt.total_seconds() / 60,
        "meal_carbs": gi["Carbs"].where(is_meal).ffill(),
        "label": label,
    }).dropna(subset=["label"])
    return out[out["glu_now"] < threshold]          # keep only rows where the question makes sense

rows5 = build_rows(all_df[all_df["subject"] == 5])
print(rows5["label"].value_counts())

In [ ]:
# Build rows for every participant and stack them into one table
parts = [build_rows(g) for _, g in all_df.groupby("subject")]
rows_all = pd.concat(parts).reset_index()   # Timestamp moves from the index into a column
print("all rows (1-minute):", rows_all.shape)

# Keep one row every 5 minutes (neighbouring minutes are almost identical)
data = rows_all[rows_all["Timestamp"].dt.minute % 5 == 0].copy()
print("rows after 5-minute sampling:", data.shape)
print("overall positive rate:", data["label"].mean().round(3))

# Per-participant check: how many rows and how many positives
per = data.groupby("subject")["label"].agg(rows="size", positives="sum", rate="mean").round(3)
print(per.describe().round(3))
print("participants with fewer than 20 positive rows:", (per["positives"] < 20).sum())

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Split by participant, not by row: test participants are never seen in training
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(data, groups=data["subject"]))
train, test = data.iloc[train_idx], data.iloc[test_idx]

train_ids, test_ids = set(train["subject"]), set(test["subject"])
print("train participants:", len(train_ids), "| test participants:", len(test_ids))
print("overlap (must be empty):", train_ids & test_ids)
print("train rows:", len(train), "| positive rate:", round(train["label"].mean(), 3))
print("test rows: ", len(test),  "| positive rate:", round(test["label"].mean(), 3))
print("test participants:", sorted(test_ids))

In [ ]:
# Static features: one value per participant, joined onto every row
static_cols = ["subject", "Age", "Gender", "BMI", "A1c PDL (Lab)",
               "Fasting GLU - PDL (Lab)", "Insulin", "HDL", "Triglycerides"]
st = bio[static_cols].copy()
st["Gender"] = (st["Gender"] == "M").astype(int)   # encode M=1, F=0
st = st.rename(columns={"A1c PDL (Lab)": "A1c", "Fasting GLU - PDL (Lab)": "fasting_glu"})

data2 = data.merge(st, on="subject", how="left")
print(data2.shape)
print("missing values per column:")
print(data2.isna().mean().round(3))
print(st.set_index("subject").loc[sorted(test_ids), ["A1c", "BMI"]].round(1))

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

# Re-create train/test from data2 using the SAME participant split
train2 = data2[data2["subject"].isin(train_ids)]
test2 = data2[data2["subject"].isin(test_ids)]

baseline_features = ["glu_now", "glu_change_30"]   # glucose only, no meal / activity / static

base = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
base.fit(train2[baseline_features], train2["label"])

pred = base.predict_proba(test2[baseline_features])[:, 1]   # probability of label 1
print("baseline AUC on unseen participants:", round(roc_auc_score(test2["label"], pred), 3))

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

glucose = ["glu_now", "glu_change_30"]
meal = ["mins_since_meal", "meal_carbs"]
static = ["Age", "Gender", "BMI", "A1c", "fasting_glu", "Insulin", "HDL", "Triglycerides"]

feature_sets = {
    "1. glucose only (baseline)": glucose,
    "2. glucose + meal": glucose + meal,
    "3. glucose + meal + static": glucose + meal + static,
}

for name, cols in feature_sets.items():
    m = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
    m.fit(train2[cols], train2["label"])
    p = m.predict_proba(test2[cols])[:, 1]
    print(f"{name:32s} AUC = {roc_auc_score(test2['label'], p):.3f}")  

In [ ]:
from sklearn.model_selection import GroupKFold

# 5-fold split by participant: every participant is in the test set exactly once
gkf = GroupKFold(n_splits=5)

for name, cols in feature_sets.items():
    aucs = []
    for tr_idx, te_idx in gkf.split(data2, groups=data2["subject"]):
        tr, te = data2.iloc[tr_idx], data2.iloc[te_idx]
        m = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
        m.fit(tr[cols], tr["label"])
        aucs.append(roc_auc_score(te["label"], m.predict_proba(te[cols])[:, 1]))
    print(f"{name:32s} mean AUC = {np.mean(aucs):.3f} (std {np.std(aucs):.3f})  folds: {[round(a, 3) for a in aucs]}")

In [ ]:
from sklearn.metrics import precision_score, recall_score

cols = glucose + meal + static

# Out-of-fold predictions: each row is predicted by a model that never saw that participant
oof = np.zeros(len(data2))
for tr_idx, te_idx in gkf.split(data2, groups=data2["subject"]):
    m = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
    m.fit(data2.iloc[tr_idx][cols], data2.iloc[tr_idx]["label"])
    oof[te_idx] = m.predict_proba(data2.iloc[te_idx][cols])[:, 1]

y = data2["label"].values
print("base rate (share of positive rows):", round(y.mean(), 3))
print(f"{'alert if p >=':>13} {'alerts':>8} {'precision':>10} {'recall':>8} {'false alarm':>12}")
for t in [0.2, 0.3, 0.4, 0.5]:
    pred = (oof >= t).astype(int)
    false_alarm = ((pred == 1) & (y == 0)).sum() / (y == 0).sum()
    print(f"{t:>13} {pred.mean():>8.2f} {precision_score(y, pred):>10.2f} {recall_score(y, pred):>8.2f} {false_alarm:>12.2f}")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

THRESHOLD = 180  # target: glucose crossing this value within the next 2 hours
root = Path("../data/cgmacros/CGMacros")

# 1. Load every participant's file and stack them into one table
frames = []
for folder in sorted(root.glob("CGMacros-*")):
    d = pd.read_csv(folder / f"{folder.name}.csv", parse_dates=["Timestamp"])
    d.columns = d.columns.str.strip()                # remove stray spaces in column names
    d["subject"] = int(folder.name.split("-")[1])    # participant id comes from the folder name
    frames.append(d)
all_df = pd.concat(frames, ignore_index=True)

# 2. Static data: one row per participant
bio = pd.read_csv(root / "bio.csv")
bio.columns = bio.columns.str.strip()

# 3. Features (past data only) and label for one participant
def build_rows(g, threshold=THRESHOLD):
    gi = g.sort_values("Timestamp").set_index("Timestamp")
    s = gi["Dexcom GL"]

    # Label: does glucose reach the threshold within the next 120 minutes?
    future_max = s[::-1].rolling(120, min_periods=60).max()[::-1].shift(-1)
    label = ((s < threshold) & (future_max >= threshold)).astype(float)
    label[s.isna() | future_max.isna()] = np.nan     # unknown label

    # A meal row has a meal type and calories > 0
    is_meal = gi["Meal Type"].notna() & (gi["Calories"] > 0)
    ts = gi.index.to_series()
    meal_time = ts.where(is_meal).ffill()            # time of the last meal

    out = pd.DataFrame({
        "subject": gi["subject"],
        "glu_now": s,
        "glu_change_30": s - s.shift(30),            # change over the last 30 minutes
        "mins_since_meal": (ts - meal_time).dt.total_seconds() / 60,
        "meal_carbs": gi["Carbs"].where(is_meal).ffill(),
        "label": label,
    }).dropna(subset=["label"])
    return out[out["glu_now"] < threshold]           # question only makes sense below the threshold

# 4. Build the table for all participants, one row every 5 minutes
rows_all = pd.concat([build_rows(g) for _, g in all_df.groupby("subject")]).reset_index()
data = rows_all[rows_all["Timestamp"].dt.minute % 5 == 0].copy()

# 5. Join static features
static_cols = ["subject", "Age", "Gender", "BMI", "A1c PDL (Lab)",
               "Fasting GLU - PDL (Lab)", "Insulin", "HDL", "Triglycerides"]
st = bio[static_cols].copy()
st["Gender"] = (st["Gender"] == "M").astype(int)     # M=1, F=0
st = st.rename(columns={"A1c PDL (Lab)": "A1c", "Fasting GLU - PDL (Lab)": "fasting_glu"})
data2 = data.merge(st, on="subject", how="left")

# 6. Feature groups and the cross-validation splitter
glucose = ["glu_now", "glu_change_30"]
meal = ["mins_since_meal", "meal_carbs"]
static = ["Age", "Gender", "BMI", "A1c", "fasting_glu", "Insulin", "HDL", "Triglycerides"]
feature_sets = {
    "1. glucose only (baseline)": glucose,
    "2. glucose + meal": glucose + meal,
    "3. glucose + meal + static": glucose + meal + static,
}
gkf = GroupKFold(n_splits=5)

print(data2.shape, "| positive rate:", round(data2["label"].mean(), 3))

In [ ]:
import joblib
from pathlib import Path

final_cols = glucose + meal + static   # same feature set as model 3

# Final model: trained on all 45 participants (evaluation numbers come from cross-validation)
final_model = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
final_model.fit(data2[final_cols], data2["label"])

Path("../models").mkdir(exist_ok=True)
joblib.dump({"model": final_model, "features": final_cols, "threshold": THRESHOLD},
            "../models/glucose_model.joblib")
print("saved. positive rate used in training:", round(data2["label"].mean(), 3))

In [ ]:
import joblib
from pathlib import Path

path = Path("../models/glucose_model.joblib")
print("size (KB):", round(path.stat().st_size / 1024, 1))

saved = joblib.load(path)
print("keys:", list(saved.keys()))
print("threshold:", saved["threshold"])
print("features:", saved["features"])
print("model type:", type(saved["model"]).__name__)

In [1]:
import sys, joblib
sys.path.insert(0, "../app")
import demo_data, model_adapter as m

print("USING_REAL_MODEL:", m.USING_REAL_MODEL)

p = demo_data.get_all_patients()[0]    # PT-101
X = m.compute_features(p["history"], p["static"], p["mins_since_meal"], p["meal_carbs"])
art = joblib.load("../models/glucose_model.joblib")

print("adapter risk:", m.predict_risk(X))
print("direct model:", round(float(art["model"].predict_proba(X[art["features"]])[0, 1]), 3))
print(X.T)

USING_REAL_MODEL: True
adapter risk: 0.906
direct model: 0.906
                     0
glu_now          170.3
glu_change_30     34.5
mins_since_meal   35.0
meal_carbs        85.0
Age               64.0
Gender             1.0
BMI               32.4
A1c                8.6
fasting_glu      125.0
Insulin           22.0
HDL               38.0
Triglycerides    210.0


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

THRESHOLD = 180  # target: glucose crossing this value within the next 2 hours
root = Path("../data/cgmacros/CGMacros")

# 1. Load every participant's file and stack them into one table
frames = []
for folder in sorted(root.glob("CGMacros-*")):
    d = pd.read_csv(folder / f"{folder.name}.csv", parse_dates=["Timestamp"])
    d.columns = d.columns.str.strip()                # remove stray spaces in column names
    d["subject"] = int(folder.name.split("-")[1])    # participant id comes from the folder name
    frames.append(d)
all_df = pd.concat(frames, ignore_index=True)

# 2. Static data: one row per participant
bio = pd.read_csv(root / "bio.csv")
bio.columns = bio.columns.str.strip()

# 3. Features (past data only) and label for one participant
def build_rows(g, threshold=THRESHOLD):
    gi = g.sort_values("Timestamp").set_index("Timestamp")
    s = gi["Dexcom GL"]

    # Label: does glucose reach the threshold within the next 120 minutes?
    future_max = s[::-1].rolling(120, min_periods=60).max()[::-1].shift(-1)
    label = ((s < threshold) & (future_max >= threshold)).astype(float)
    label[s.isna() | future_max.isna()] = np.nan     # unknown label

    # A meal row has a meal type and calories > 0
    is_meal = gi["Meal Type"].notna() & (gi["Calories"] > 0)
    ts = gi.index.to_series()
    meal_time = ts.where(is_meal).ffill()            # time of the last meal

    out = pd.DataFrame({
        "subject": gi["subject"],
        "glu_now": s,
        "glu_change_30": s - s.shift(30),            # change over the last 30 minutes
        "mins_since_meal": (ts - meal_time).dt.total_seconds() / 60,
        "meal_carbs": gi["Carbs"].where(is_meal).ffill(),
        "label": label,
    }).dropna(subset=["label"])
    return out[out["glu_now"] < threshold]           # question only makes sense below the threshold

# 4. Build the table for all participants, one row every 5 minutes
rows_all = pd.concat([build_rows(g) for _, g in all_df.groupby("subject")]).reset_index()
data = rows_all[rows_all["Timestamp"].dt.minute % 5 == 0].copy()

# 5. Join static features
static_cols = ["subject", "Age", "Gender", "BMI", "A1c PDL (Lab)",
               "Fasting GLU - PDL (Lab)", "Insulin", "HDL", "Triglycerides"]
st = bio[static_cols].copy()
st["Gender"] = (st["Gender"] == "M").astype(int)     # M=1, F=0
st = st.rename(columns={"A1c PDL (Lab)": "A1c", "Fasting GLU - PDL (Lab)": "fasting_glu"})
data2 = data.merge(st, on="subject", how="left")

# 6. Feature groups and the cross-validation splitter
glucose = ["glu_now", "glu_change_30"]
meal = ["mins_since_meal", "meal_carbs"]
static = ["Age", "Gender", "BMI", "A1c", "fasting_glu", "Insulin", "HDL", "Triglycerides"]
feature_sets = {
    "1. glucose only (baseline)": glucose,
    "2. glucose + meal": glucose + meal,
    "3. glucose + meal + static": glucose + meal + static,
}
gkf = GroupKFold(n_splits=5)

print(data2.shape, "| positive rate:", round(data2["label"].mean(), 3))

(107269, 15) | positive rate: 0.157


In [3]:
# Rows in the training table that look like the demo patient:
# glucose close to the threshold and rising quickly over the last 30 minutes
similar = data2[data2["glu_now"].between(160, 180) & (data2["glu_change_30"] > 25)]
print("similar rows:", len(similar), "| participants:", similar["subject"].nunique())
print("share that crossed 180 within 2 hours:", round(similar["label"].mean(), 3))

similar rows: 1918 | participants: 45
share that crossed 180 within 2 hours: 0.561


In [4]:
# Narrower match: also a similar meal and similar A1c to the demo patient (PT-101)
near = data2[
    data2["glu_now"].between(160, 180)
    & (data2["glu_change_30"] > 25)
    & (data2["meal_carbs"] >= 60)
    & (data2["mins_since_meal"] <= 60)
    & (data2["A1c"] >= 7.5)
]
print("rows:", len(near), "| participants:", near["subject"].nunique())
print("share that crossed 180:", round(near["label"].mean(), 3) if len(near) else "no rows")

rows: 11 | participants: 2
share that crossed 180: 1.0


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

THRESHOLD = 180  # target: glucose crossing this value within the next 2 hours
root = Path("../data/cgmacros/CGMacros")

# 1. Load every participant's file and stack them into one table
frames = []
for folder in sorted(root.glob("CGMacros-*")):
    d = pd.read_csv(folder / f"{folder.name}.csv", parse_dates=["Timestamp"])
    d.columns = d.columns.str.strip()                # remove stray spaces in column names
    d["subject"] = int(folder.name.split("-")[1])    # participant id comes from the folder name
    frames.append(d)
all_df = pd.concat(frames, ignore_index=True)

# 2. Static data: one row per participant
bio = pd.read_csv(root / "bio.csv")
bio.columns = bio.columns.str.strip()

# 3. Features (past data only) and label for one participant
def build_rows(g, threshold=THRESHOLD):
    gi = g.sort_values("Timestamp").set_index("Timestamp")
    s = gi["Dexcom GL"]

    # Label: does glucose reach the threshold within the next 120 minutes?
    future_max = s[::-1].rolling(120, min_periods=60).max()[::-1].shift(-1)
    label = ((s < threshold) & (future_max >= threshold)).astype(float)
    label[s.isna() | future_max.isna()] = np.nan     # unknown label

    # A meal row has a meal type and calories > 0
    is_meal = gi["Meal Type"].notna() & (gi["Calories"] > 0)
    ts = gi.index.to_series()
    meal_time = ts.where(is_meal).ffill()            # time of the last meal

    out = pd.DataFrame({
        "subject": gi["subject"],
        "glu_now": s,
        "glu_change_30": s - s.shift(30),            # change over the last 30 minutes
        "mins_since_meal": (ts - meal_time).dt.total_seconds() / 60,
        "meal_carbs": gi["Carbs"].where(is_meal).ffill(),
        "label": label,
    }).dropna(subset=["label"])
    return out[out["glu_now"] < threshold]           # question only makes sense below the threshold

# 4. Build the table for all participants, one row every 5 minutes
rows_all = pd.concat([build_rows(g) for _, g in all_df.groupby("subject")]).reset_index()
data = rows_all[rows_all["Timestamp"].dt.minute % 5 == 0].copy()

# 5. Join static features
static_cols = ["subject", "Age", "Gender", "BMI", "A1c PDL (Lab)",
               "Fasting GLU - PDL (Lab)", "Insulin", "HDL", "Triglycerides"]
st = bio[static_cols].copy()
st["Gender"] = (st["Gender"] == "M").astype(int)     # M=1, F=0
st = st.rename(columns={"A1c PDL (Lab)": "A1c", "Fasting GLU - PDL (Lab)": "fasting_glu"})
data2 = data.merge(st, on="subject", how="left")

# 6. Feature groups and the cross-validation splitter
glucose = ["glu_now", "glu_change_30"]
meal = ["mins_since_meal", "meal_carbs"]
static = ["Age", "Gender", "BMI", "A1c", "fasting_glu", "Insulin", "HDL", "Triglycerides"]
feature_sets = {
    "1. glucose only (baseline)": glucose,
    "2. glucose + meal": glucose + meal,
    "3. glucose + meal + static": glucose + meal + static,
}
gkf = GroupKFold(n_splits=5)

print(data2.shape, "| positive rate:", round(data2["label"].mean(), 3))

(107269, 15) | positive rate: 0.157


In [2]:
# Out-of-fold predictions from model 3: each row is predicted by a model that never saw that participant
cols = glucose + meal + static
oof = np.zeros(len(data2))
for tr_idx, te_idx in gkf.split(data2, groups=data2["subject"]):
    m = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
    m.fit(data2.iloc[tr_idx][cols], data2.iloc[tr_idx]["label"])
    oof[te_idx] = m.predict_proba(data2.iloc[te_idx][cols])[:, 1]

# Group rows by predicted probability and compare with what actually happened
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]
tmp = pd.DataFrame({"pred": oof, "actual": data2["label"].values})
tmp["bin"] = pd.cut(tmp["pred"], bins=bins, include_lowest=True)
table = tmp.groupby("bin", observed=True).agg(
    rows=("actual", "size"),
    mean_pred=("pred", "mean"),
    actual_rate=("actual", "mean"),
).round(3)
print(table)

                rows  mean_pred  actual_rate
bin                                         
(-0.001, 0.1]  58901      0.036        0.067
(0.1, 0.2]     19915      0.145        0.150
(0.2, 0.3]     10223      0.245        0.202
(0.3, 0.4]      5879      0.345        0.273
(0.4, 0.5]      3681      0.446        0.379
(0.5, 0.6]      2708      0.547        0.438
(0.6, 0.8]      3912      0.693        0.540
(0.8, 1.0]      2050      0.869        0.730


In [3]:
import json

# Save the validation table so the dashboard can show "how often this score actually crossed 180"
lookup = [
    {"low": float(iv.left), "high": float(iv.right),
     "rows": int(r["rows"]), "actual_rate": float(r["actual_rate"])}
    for iv, r in table.iterrows()
]
with open("../models/calibration_table.json", "w") as f:
    json.dump(lookup, f, indent=2)
print(lookup[-1])

{'low': 0.8, 'high': 1.0, 'rows': 2050, 'actual_rate': 0.73}
